# Obligatorio Taller de Tecnologías 2: Agentic RAG

Universidad ORT Uruguay, 2026.

Este notebook integra y demuestra los módulos de `tt2_agentic_rag`. La lógica vive en `src/`; acá solo se usa.
La versión autocontenida para Google Colab es `notebooks/obligatorio_colab.ipynb`.

Requisitos: `pip install -e .` y un `.env` con `GROQ_API_KEY` y `PINECONE_API_KEY` (ver `.env.example`).

## 1. Configuración

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

import tt2_agentic_rag

# Raíz del repo, tanto si el kernel arranca en notebooks/ como en la raíz
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
load_dotenv(ROOT / ".env")

PAPERS_DIR = ROOT / (os.getenv("PAPERS_DIR") or "data/papers")
DATASET_PATH = ROOT / (os.getenv("DATASET_PATH") or "data/datasets/netflix_titles.csv")
MEMORY_DB_PATH = ROOT / (os.getenv("MEMORY_DB_PATH") or "storage/memory.sqlite")
LLM_MODEL = (os.getenv("LLM_MODEL") or "qwen/qwen3.8-27b")

for key in ("GROQ_API_KEY", "PINECONE_API_KEY"):
    if not os.getenv(key):
        raise ValueError(f"Falta {key} en .env (ver .env.example).")

print("tt2_agentic_rag", tt2_agentic_rag.__version__)
print("Papers: ", PAPERS_DIR)
print("Dataset:", DATASET_PATH)

tt2_agentic_rag 0.1.0
Papers:  /home/guille/tt2-agentic-rag/data/papers
Dataset: /home/guille/tt2-agentic-rag/data/datasets/netflix_titles.csv


## 2. Ingesta de papers

PDFs → páginas (`PyPDFLoader`) → chunks de 800 caracteres con solapamiento de 120 (`RecursiveCharacterTextSplitter`)
→ normalización de espacios → embeddings `all-MiniLM-L6-v2` → índice `research-papers` en Pinecone.

Cada chunk guarda `file_name` y `page` para citar la fuente. Si el índice ya tiene vectores no se reindexa
(`force=True` reindexa con IDs deterministas, sin duplicar).

In [2]:
from tt2_agentic_rag import ingest

raw_papers_docs = ingest.load_pdfs(PAPERS_DIR)
paper_chunks = ingest.split_documents(raw_papers_docs)

print(f"\nPáginas originales: {len(raw_papers_docs)}")
print(f"Chunks generados y limpios: {len(paper_chunks)}")
sample = paper_chunks[0]
print(f"\nMuestra (Paper: {sample.metadata['file_name']}, Pág: {sample.metadata['page']}):")
print(sample.page_content[:200] + "...")

Cargado: 2507.14805v1.pdf (34 páginas)


Cargado: Attention is all you need (1).pdf (15 páginas)


Cargado: Designing Data-Intensive Applications The Big Ideas Behind Reliable, Scalable, and Maintainable Systems by Martin Kleppmann (z-l.pdf (613 páginas)


Cargado: GPT Improving Language Understanding by Generative Pretraining.pdf (12 páginas)


Cargado: Language Models are Few-Shot Learners (GPT3).pdf (75 páginas)


Cargado: Scaling Laws for Neural Language Models.pdf (30 páginas)

Páginas originales: 779
Chunks generados y limpios: 3121

Muestra (Paper: 2507.14805v1.pdf, Pág: 1):
SUBLIMINAL LEARNING : LANGUAGE MODELS TRANSMIT BEHAVIORAL TRAITS VIA HIDDEN SIGNALS IN DATA Alex Cloud∗1, Minh Le∗1 James Chua2, Jan Betley2, Anna Sztyber-Betley3, Jacob Hilton4 Samuel Marks5, Owain E...


In [3]:
vectorstore_papers = ingest.build_index(PAPERS_DIR)
print("Vectores en 'research-papers':", ingest.index_vector_count(ingest.PAPERS_INDEX_NAME))

/home/guille/tt2-agentic-rag/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 20818.92it/s]

El índice 'research-papers' ya tiene 3121 vectores. Conectando sin reindexar.


Vectores en 'research-papers': 3121


## 3. Recuperación semántica (RAG)

In [4]:
from tt2_agentic_rag import rag

for doc in rag.search("attention mechanism transformer architecture", k=3):
    print(f"[{doc.metadata['file_name']} | pág. {ingest.as_int(doc.metadata['page'])}] {doc.page_content[:150]}...")

[Attention is all you need (1).pdf | pág. 3] Figure 1: The Transformer - model architecture. The Transformer follows this overall architecture using stacked self-attention and point-wise, fully c...
[Attention is all you need (1).pdf | pág. 5] is similar to that of single-head attention with full dimensionality. 3.2.3 Applications of Attention in our Model The Transformer uses multi-head att...
[Attention is all you need (1).pdf | pág. 2] tion models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]. In all but ...


In [5]:
# La misma búsqueda, a través de la tool que usa el agente
print(rag.query_research_papers.invoke("scaling laws compute data model size")[:1200])

[Documento: Scaling Laws for Neural Language Models.pdf | Página: 18]
dataset size with model size, whereas we ﬁnd a sub-linear scaling. There are some parallels between our ﬁndings on optimal allocation of compute and [Kom19], including power-law learning curves. EfﬁcientNets [TL19] also appear to obey an approximate power-law relation between accuracy and model size. Very recent work [RRBS19b] studies scaling with both dataset size and model size for a variety of datasets, and ﬁts an ansatz similar to ours. EfﬁcientNet [TL19] advocates scaling depth and width exponentially (with different coefﬁcients) for optimal performance of image models, resulting in a power-law scaling of width as a function of depth. We ﬁnd that for language models this power should be roughly one when scaling up (as width/depth should remain ﬁxed).
---
[Documento: Scaling Laws for Neural Language Models.pdf | Página: 18]
7 Related Work Power laws can arise from a wide variety of sources [THK18]. Power-law scal

## 4. Dataset de Kaggle: Netflix Titles

Limpieza: nulos de texto → cadena vacía y colapso de espacios, sin tocar el vocabulario. Título, tipo, géneros,
director, elenco y descripción se concatenan con etiquetas (`Title: ...`) en `text_for_embedding`; país, año y tipo
quedan como metadatos filtrables. Índice `netflix-catalog` en Pinecone, con `show_id` como ID.

In [6]:
from tt2_agentic_rag import dataset

df_netflix = dataset.load_dataset(DATASET_PATH)
print(f"Filas: {df_netflix.shape[0]} | Columnas: {df_netflix.shape[1]}")

df_netflix = dataset.prepare_dataset(df_netflix)
print("\nMuestra de texto preparado para embeddings:")
print(df_netflix["text_for_embedding"].iloc[0])
df_netflix[["title", "type", "country", "release_year"]].head(3)

Filas: 8807 | Columnas: 12

Muestra de texto preparado para embeddings:
Title: Dick Johnson Is Dead. Type: Movie. Genres: Documentaries. Director: Kirsten Johnson. Cast: . Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.


,title,type,country,release_year
0,Dick Johnson Is Dead,Movie,United States,2020
1,Blood & Water,TV Show,South Africa,2021
2,Ganglands,TV Show,Unknown,2021


In [7]:
vectorstore_netflix = dataset.build_index(df_netflix)
print("Vectores en 'netflix-catalog':", ingest.index_vector_count(dataset.NETFLIX_INDEX_NAME))

El índice 'netflix-catalog' ya tiene 8807 vectores. Conectando sin reindexar.


Vectores en 'netflix-catalog': 8807


In [8]:
def mostrar(docs):
    for i, doc in enumerate(docs, start=1):
        m = doc.metadata
        print(f"  {i}. {m.get('title')} ({m.get('type')}) | {m.get('country')} | {ingest.as_int(m.get('release_year'))}")


print("TEST 1: búsqueda semántica (trama)")
mostrar(dataset.search("detective solving murder mystery in a small dark town", k=3))

print("\nTEST 2: búsqueda conceptual (ciencia ficción)")
mostrar(dataset.search("space travel and exploration of alien civilizations", k=3))

print("\nTEST 3: filtro por metadatos (películas de España)")
mostrar(dataset.search(
    "crime and thrilling heist", k=3,
    filter={"country": {"$eq": "Spain"}, "type": {"$eq": "Movie"}},
))

TEST 1: búsqueda semántica (trama)


  1. Dark Crimes (Movie) | United Kingdom, Poland, United States | 2016
  2. Small Town Crime (Movie) | United States | 2017
  3. The Murder Detectives (TV Show) | United Kingdom | 2015

TEST 2: búsqueda conceptual (ciencia ficción)
  1. The Search for Life in Space (Movie) | United States | 2016
  2. Ancient Aliens (TV Show) | United States | 2018
  3. Alien Contact: Outer Space (Movie) | United States | 2017

TEST 3: filtro por metadatos (películas de España)


  1. Money Heist: The Phenomenon (Movie) | Spain | 2020
  2. Twin Murders: the Silence of the White City (Movie) | Spain | 2020
  3. The Occupant (Movie) | Spain | 2020


## 5. Agente LangGraph

Grafo armado a mano en `agent.build_agent` (`START → llm → tools → llm … → END`, sin `create_agent`,
`create_react_agent`, `ToolNode` ni `tools_condition`). El LLM decide con tool calling entre:

- `query_research_papers` (papers),
- `query_netflix_catalog` (dataset),
- `get_user_memory` (memoria a largo plazo del usuario).

In [9]:
from langchain_groq import ChatGroq

from tt2_agentic_rag.agent import build_agent
from tt2_agentic_rag.dataset import query_netflix_catalog
from tt2_agentic_rag.memory import LongTermMemory, get_checkpointer, get_store
from tt2_agentic_rag.rag import query_research_papers

llm = ChatGroq(model=LLM_MODEL, temperature=0, max_tokens=500)

checkpointer = get_checkpointer(MEMORY_DB_PATH)  # memoria a corto plazo (por thread_id)
store = get_store(MEMORY_DB_PATH)
long_term_memory = LongTermMemory(llm)           # hechos del usuario (por user_id)

agent = build_agent(
    llm=llm,
    tools=[query_netflix_catalog, query_research_papers, long_term_memory.as_tool()],
    checkpointer=checkpointer,
    store=store,
)
print("¡Agente compilado!")

¡Agente compilado!


In [10]:
# Estructura del grafo (Mermaid)
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm(llm)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm;
	llm -.-> __end__;
	llm -.-> tools;
	tools --> llm;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 6. Memoria y conversaciones múltiples

- **Corto plazo**: el checkpointer SQLite guarda el historial de cada `thread_id`; sobrevive a reinicios del kernel.
- **Largo plazo**: al cierre de cada turno, `LongTermMemory.save_from_turn` extrae hechos del usuario con el LLM,
  descarta los que quedan contradichos y guarda el nuevo en Pinecone (`user-memory`, filtrado por `user_id`).
  En otra conversación el agente los recupera con `get_user_memory`.

Cada ejecución del notebook usa IDs de hilo nuevos para que la demo arranque de cero.

In [11]:
import uuid

from langchain_core.messages import AIMessage, HumanMessage

RUN = uuid.uuid4().hex[:6]


def conversar(mensaje: str, thread_id: str, user_id: str = "nicolas") -> None:
    config = {"configurable": {"thread_id": f"{thread_id}-{RUN}", "user_id": user_id}}
    print(f"\n🧑 [{user_id} | {thread_id}]: {mensaje}")

    previos = len(agent.get_state(config).values.get("messages", []))
    response = agent.invoke({"messages": [HumanMessage(content=mensaje)]}, config=config)
    nuevos = response["messages"][previos:]

    tools = [c["name"] for m in nuevos if isinstance(m, AIMessage) for c in m.tool_calls]
    print(f"🔧 Herramientas: {', '.join(tools) or 'ninguna'}")
    bot_reply = response["messages"][-1]
    print(f"🤖 Asistente: {bot_reply.content}")

    # Memoria a largo plazo al cierre del turno
    long_term_memory.save_from_turn([HumanMessage(content=mensaje), bot_reply], user_id=user_id)
    print("-" * 75)

## 7. Demostración y conclusiones

In [12]:
# 1. Small-talk (no debe disparar herramientas)
conversar("¡Hola! ¿Cómo andás? Me llamo Nicolás y me gustan mucho los thrillers psicológicos.", thread_id="hilo-1")


🧑 [nicolas | hilo-1]: ¡Hola! ¿Cómo andás? Me llamo Nicolás y me gustan mucho los thrillers psicológicos.


🔧 Herramientas: ninguna
🤖 Asistente: ¡Hola, Nicolás! Muy bien, gracias por preguntar. Qué gusto que te gusten los thrillers psicológicos, es un género fascinante.

Si querés, puedo buscar en el catálogo de Netflix algunas recomendaciones de thrillers psicológicos que te puedan interesar. ¿Te parece?


  [Memoria LTM guardada para 'nicolas']: Nicolás disfruta mucho de los thrillers psicológicos.
---------------------------------------------------------------------------


In [13]:
# 2. Consulta a los papers (RAG)
conversar("¿Qué explican los documentos sobre la arquitectura Transformer y el encoder?", thread_id="hilo-1")


🧑 [nicolas | hilo-1]: ¿Qué explican los documentos sobre la arquitectura Transformer y el encoder?


🔧 Herramientas: query_research_papers
🤖 Asistente: Según los documentos, la arquitectura Transformer y su encoder se describen principalmente en el paper **"Attention is all you need"** (Vaswani et al.). Esto es lo que explican:

## Arquitectura general
El Transformer sigue una arquitectura general que utiliza **capas apiladas de auto-atención (self-attention) y capas totalmente conectadas punto a punto (point-wise fully connected)** tanto para el encoder como para el decoder (Figura 1 del paper, pág. 3).

## El Encoder
- El encoder está compuesto por una **pila de N = 6 capas idénticas**.
- Cada capa tiene **dos sub-capas**:
  1. Un mecanismo de **auto-atención multi-cabeza (multi-head self-attention)**.
  2. Una **red feed-forward totalmente conectada** punto a punto (position-wise).
- Se emplea una **conexión residual** alrededor de cada una de las dos sub-capas, seguida de **normalización de capa (layer normalization)**. Es decir, la salida de cada sub-capa es `LayerNorm(x + Sublay

---------------------------------------------------------------------------


In [14]:
# 3. Memoria a corto plazo (referencia implícita al turno anterior en el mismo hilo)
conversar("¿En qué página del paper estaba eso que me acabás de resumir?", thread_id="hilo-1")


🧑 [nicolas | hilo-1]: ¿En qué página del paper estaba eso que me acabás de resumir?


🔧 Herramientas: ninguna
🤖 Asistente: Lo que te resumí proviene de varias páginas del paper **"Attention is all you need"**:

- **Página 3**: la arquitectura general del Transformer y la descripción del encoder (pila de N = 6 capas, dos sub-capas por capa, conexiones residuales y layer normalization).
- **Página 5**: las tres formas en que el modelo usa la atención multi-cabeza, incluyendo las capas de auto-atención del encoder y la atención encoder-decoder.
- **Página 8**: los detalles sobre dropout (tasa `Pdrop = 0.1`) y label smoothing.

Así que no era de una sola página, sino de esas tres. ¿Querés que profundice en alguna de ellas?


---------------------------------------------------------------------------


In [15]:
# 4. Consulta al dataset de Netflix (RAG)
conversar("Buscame películas o series en Netflix relacionadas con detectives e investigación criminal.", thread_id="hilo-1")


🧑 [nicolas | hilo-1]: Buscame películas o series en Netflix relacionadas con detectives e investigación criminal.


🔧 Herramientas: query_netflix_catalog
🤖 Asistente: Acá van algunas opciones de Netflix relacionadas con detectives e investigación criminal:

**The Investigator: A British Crime Story** (2018, Reino Unido)
Docuseries de crimen. Tras 40 años de evidencia inconclusa, el reconocido criminólogo Mark Williams-Thomas reexamina una serie de asesinatos misteriosos.

**Crime Stories: India Detectives** (2021)
Docuseries de crimen. Cámaras que siguen a la policía de Bengaluru en acción ofrecen una mirada poco común a los complejos y desafiantes entresijos de cuatro grandes investigaciones criminales.

**Unsolved Mysteries** (2020)
Docuseries de crimen. Casos reales de desapariciones desconcertantes, asesinatos impactantes y encuentros paranormales impulsan esta apasionante reedición de la icónica serie documental.

**A Perfect Crime** (2020, Alemania)
Docuseries de crimen e internacional. Investiga el asesinato de 1991 del político Detlev Rohwedder, un misterio sin resolver en el corazón de la t

  [Memoria LTM guardada para 'nicolas']: El usuario prefiere consumir contenido audiovisual en la plataforma Netflix y tiene interés en el género de detectives e investigación criminal.
---------------------------------------------------------------------------


In [16]:
# 5. Memoria a largo plazo: hilo nuevo, mismo user_id
conversar("Hola de nuevo, arrancamos un chat nuevo. ¿Te acordás cómo me llamo y qué género de películas me gusta?", thread_id="hilo-2")


🧑 [nicolas | hilo-2]: Hola de nuevo, arrancamos un chat nuevo. ¿Te acordás cómo me llamo y qué género de películas me gusta?


🔧 Herramientas: get_user_memory
🤖 Asistente: ¡Hola de nuevo, Nicolás! Sí, me acuerdo: te gustan los **thrillers psicológicos**, y también el contenido de Netflix relacionado con **detectives e investigación criminal**.

¿Querés que te busque alguna recomendación de ese estilo en el catálogo?


  [Memoria LTM guardada para 'nicolas']: El usuario se llama Nicolás y le gustan los thrillers psicológicos, así como el contenido de Netflix relacionado con detectives e investigación criminal.
---------------------------------------------------------------------------


In [17]:
# 6. Control de alucinaciones (información inexistente en las fuentes)
conversar("¿Qué dice el paper de Attention Is All You Need sobre el cultivo de yerba mate en Uruguay?", thread_id="hilo-2")


🧑 [nicolas | hilo-2]: ¿Qué dice el paper de Attention Is All You Need sobre el cultivo de yerba mate en Uruguay?


🔧 Herramientas: query_research_papers
🤖 Asistente: No, Nicolás. El paper *Attention Is All You Need* no dice nada sobre el cultivo de yerba mate en Uruguay.

Ese trabajo es el paper original del modelo **Transformer** (Vaswani et al., 2017) y trata sobre arquitectura de modelos de lenguaje, mecanismos de atención, entrenamiento y evaluación en tareas de traducción y modelado de lenguaje. No tiene relación con agricultura, yerba mate ni Uruguay.

La búsqueda en la colección de papers no devolvió ningún contenido sobre yerba mate; los resultados que aparecieron corresponden a otro documento (*Language Models are Few-Shot Learners* / GPT-3), que tampoco aborda ese tema.

Si querés, puedo buscar en la colección algún paper que sí trate de modelos de atención o de arquitectura Transformer.


---------------------------------------------------------------------------


In [18]:
# Las conversaciones son independientes: hilo-1 tiene su propio historial
for hilo in ("hilo-1", "hilo-2"):
    state = agent.get_state({"configurable": {"thread_id": f"{hilo}-{RUN}"}})
    print(f"{hilo}: {len(state.values['messages'])} mensajes guardados en SQLite")

hilo-1: 12 mensajes guardados en SQLite
hilo-2: 8 mensajes guardados en SQLite


## 8. Interfaz web con Gradio (opcional)

Levanta una interfaz web interactiva reutilizando el `agent` y la `long_term_memory` ya inicializados.

In [19]:
from tt2_agentic_rag.ui import create_demo

demo = create_demo(
    agent,
    save_memory_fn=lambda msgs, uid: long_term_memory.save_from_turn(msgs, user_id=uid),
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
